# DATAFEST: submissions.csv
Regresión logística con 8 variables continuas y 7 reglas, entrenada con toda la data (enero–noviembre) para predecir diciembre.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
print(train.shape, test.shape)

(110100, 25) (9900, 24)


In [ ]:
CONTINUAS = ["dias_ultima_transaccion", "numero_productos", "ratio_deuda_ingresos", "ingresos",
             "saldo_promedio", "antiguedad_cuenta_meses", "distancia_sucursal_km", "edad"]
FLAGS = ["riesgo_low_productos_3_o_mas", "riesgo_low_movil_tarjeta_2_o_mas", "riesgo_high_transaccion_mas_180",
         "riesgo_high_canal_branch", "riesgo_high_branch_transaccion_mas_180", "movil_y_tarjeta", "transaccion_mas_180"]
VARS = CONTINUAS + FLAGS


def agregar_flags(d):
    d = d.copy()
    low = d["banda_riesgo"].eq("low")
    high = d["banda_riesgo"].eq("high")
    mt = d["activo_movil"].astype(bool) & d["tiene_tarjeta_credito"].astype(bool)
    t180 = d["dias_ultima_transaccion"] > 180
    br = d["canal_adquisicion"].eq("branch")
    d["movil_y_tarjeta"] = mt.astype(int)
    d["riesgo_low_productos_3_o_mas"] = (low & (d["numero_productos"] >= 3)).astype(int)
    d["riesgo_low_movil_tarjeta_2_o_mas"] = (low & (d["numero_productos"] >= 2) & mt).astype(int)
    d["transaccion_mas_180"] = t180.astype(int)
    d["riesgo_high_transaccion_mas_180"] = (high & t180).astype(int)
    d["riesgo_high_canal_branch"] = (high & br).astype(int)
    d["riesgo_high_branch_transaccion_mas_180"] = (high & br & t180).astype(int)
    return d


X_train = agregar_flags(train)[VARS]
X_test = agregar_flags(test)[VARS]
y_train = train["objetivo"]

In [ ]:
modelo = make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=5000))
modelo.fit(X_train, y_train)

submissions = pd.DataFrame({"id_cliente": test["id_cliente"], "prediccion": modelo.predict_proba(X_test)[:, 1]})
submissions.to_csv("submissions.csv", index=False)

In [ ]:
assert list(submissions.columns) == ["id_cliente", "prediccion"]
assert len(submissions) == len(test) and (submissions["id_cliente"].values == test["id_cliente"].values).all()
assert submissions["prediccion"].between(0, 1).all() and submissions["prediccion"].notna().all()
print(submissions.shape)
submissions.head()

(9900, 2)


,id_cliente,prediccion
0,2,0.128301
1,3,0.054987
2,7,0.135968
3,10,0.059035
4,11,0.133056


## Resultados de validación
Cada mes se predice entrenando solo con los meses anteriores. Gini = 2 · AUC − 1.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr


def gini(y, p):
    return 2 * roc_auc_score(y, p) - 1


def nuevo_modelo():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=5000))


df = agregar_flags(train).sort_values(["mes", "id_cliente"]).reset_index(drop=True)
y = df["objetivo"].values
meses = [202606, 202607, 202608, 202609, 202610, 202611]

In [ ]:
filas = []
for m in meses:
    tr, va = (df["mes"] < m).values, (df["mes"] == m).values
    p = nuevo_modelo().fit(df.loc[tr, VARS], y[tr]).predict_proba(df.loc[va, VARS])[:, 1]
    filas.append({"mes": m, "entrena_con": f"{df.loc[tr, 'mes'].min()}–{df.loc[tr, 'mes'].max()}", "gini": gini(y[va], p)})
gini_mes = pd.DataFrame(filas).set_index("mes")
print(f"Media 6 meses (jun–nov): {gini_mes['gini'].mean():.4f}")
print(f"Media sep–nov:           {gini_mes.loc[[202609, 202610, 202611], 'gini'].mean():.4f}")
gini_mes.round(4)

Media 6 meses (jun–nov): 0.2639
Media sep–nov:           0.2557


,entrena_con,gini
mes,,
202606,202601–202605,0.2678
202607,202601–202606,0.2736
202608,202601–202607,0.2748
202609,202601–202608,0.2526
202610,202601–202609,0.2633
202611,202601–202610,0.2513


## Entrenar hasta septiembre, validar en octubre, re-entrenar hasta octubre y probar en noviembre

In [ ]:
fit, val, nov = df["mes"] <= 202609, df["mes"] == 202610, df["mes"] == 202611
m1 = nuevo_modelo().fit(df.loc[fit, VARS], y[fit])
m2 = nuevo_modelo().fit(df.loc[fit | val, VARS], y[fit | val])
print(f"Validación octubre (entrena ene–sep):  {gini(y[val], m1.predict_proba(df.loc[val, VARS])[:, 1]):.4f}")
print(f"Test noviembre (entrena ene–oct):      {gini(y[nov], m2.predict_proba(df.loc[nov, VARS])[:, 1]):.4f}")

Validación octubre (entrena ene–sep):  0.2633
Test noviembre (entrena ene–oct):      0.2513


## Reglas extra del amigo
Se agregan al modelo de 15 variables y se compara el Gini medio de 6 meses.

In [ ]:
low, high = df["banda_riesgo"].eq("low"), df["banda_riesgo"].eq("high")
df["riesgo_low_deuda_06"] = (low & (df["ratio_deuda_ingresos"] >= 0.6)).astype(int)
df["riesgo_high_menos_100"] = (high & (df["dias_ultima_transaccion"] < 100)).astype(int)

variantes = {
    "LR 15 (actual)": VARS,
    "+ low deuda >= 0.6": VARS + ["riesgo_low_deuda_06"],
    "+ high < 100 días": VARS + ["riesgo_high_menos_100"],
    "+ ambas (17 vars)": VARS + ["riesgo_low_deuda_06", "riesgo_high_menos_100"],
}
filas = []
for nombre, cols in variantes.items():
    fila = {"modelo": nombre}
    for m in meses:
        tr, va = (df["mes"] < m).values, (df["mes"] == m).values
        fila[m] = gini(y[va], nuevo_modelo().fit(df.loc[tr, cols], y[tr]).predict_proba(df.loc[va, cols])[:, 1])
    filas.append(fila)
extra = pd.DataFrame(filas).set_index("modelo")
extra["media_6m"] = extra[meses].mean(axis=1)
extra["sep_nov"] = extra[[202609, 202610, 202611]].mean(axis=1)
extra["dif_vs_actual"] = extra["media_6m"] - extra.loc["LR 15 (actual)", "media_6m"]
extra.round(4)

,202606,202607,202608,202609,202610,202611,media_6m,sep_nov,dif_vs_actual
modelo,,,,,,,,,
LR 15 (actual),0.2678,0.2736,0.2748,0.2526,0.2633,0.2513,0.2639,0.2557,0.0000
+ low deuda >= 0.6,0.2677,0.2737,0.2755,0.2530,0.2640,0.2501,0.2640,0.2557,0.0001
+ high < 100 días,0.2678,0.2724,0.2739,0.2532,0.2632,0.2521,0.2638,0.2562,-0.0001
+ ambas (17 vars),0.2676,0.2727,0.2746,0.2534,0.2639,0.2507,0.2638,0.2560,-0.0001


In [ ]:
hh = df[high]
print("High, conversión por tramo de días:", hh.groupby(pd.cut(hh["dias_ultima_transaccion"], [0, 99, 179, 400]), observed=True)["objetivo"].mean().round(3).to_dict())
ll = df[low]
print("Low, conversión con deuda >= 0.6:", ll.groupby(ll["ratio_deuda_ingresos"] >= 0.6)["objetivo"].mean().round(3).to_dict())

High, conversión por tramo de días: {Interval(0, 99, closed='right'): 0.154, Interval(99, 179, closed='right'): 0.126, Interval(179, 400, closed='right'): 0.049}
Low, conversión con deuda >= 0.6: {False: 0.188, True: 0.162}
